# Notebook to clean up data to be used in workload projections

In [10]:
spark

In [11]:
username = spark.sparkContext.sparkUser()
print(username)

catarinap


#### Imports & others

In [12]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import DataFrame, SparkSession
import os

In [13]:
output_directory = f'/user/{username}/notebooks/Workload Projections'
print(output_directory)

/user/catarinap/notebooks/Workload Projections


## Salesforce data

### Queue Mapping logic
https://looker.is.adyen.com/projects/spark/files/support_tooling/views/dt_queue_mapping.view.lkml

In [14]:
def create_queue_mapping_salesforce_df(spark: SparkSession):
    schema = T.StructType([
        T.StructField("queue", T.StringType(), nullable=False),
        T.StructField("team", T.StringType(), nullable=False),
        T.StructField("subdomain", T.StringType(), nullable=False),
        T.StructField("domain", T.StringType(), nullable=False),
        T.StructField("department", T.StringType(), nullable=False)
    ])
    
    data = [
        ('AML', 'Anti Money Laundering', 'Anti Money Laundering', 'Merchant Operations', 'Operations'),
        ('AntiMoneyLaunderingAML', 'Anti Money Laundering', 'Anti Money Laundering', 'Merchant Operations', 'Operations'),
        ('CustomerRiskReview', 'Customer Risk Review', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('CustomerRiskReviewUnclassified', 'Customer Risk Review', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('PlatformsKYCOperations', 'KYC Operations', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('Onboarding', 'Onboarding', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('OnboardingUnclassified', 'Onboarding', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('Underwriting', 'Underwriting', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('UnderwritingUnclassified', 'Underwriting', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('CreditRisk', 'Credit Risk', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('MerchantFraud', 'Merchant Fraud', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('MerchantFraudUnclassified', 'Merchant Fraud', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('MerchantPotentialLiabilityMPL', 'Merchant Potential Liability', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('MPLUnclassified', 'Merchant Potential Liability', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('PaymentCardIndustryOperationsPCI', 'Payment Card Industry Operations', 'Scheme Operations', 'Merchant Operations', 'Operations'),
        ('PCI', 'Payment Card Industry Operations', 'Scheme Operations', 'Merchant Operations', 'Operations'),
        ('PCIOperationsUnclassified', 'Payment Card Industry Operations', 'Scheme Operations', 'Merchant Operations', 'Operations'),
        ('Scheme Operations', 'Scheme Operations', 'Scheme Operations', 'Merchant Operations', 'Operations'),
        ('SchemeOperations', 'Scheme Operations', 'Scheme Operations', 'Merchant Operations', 'Operations'),
        ('SchemeOperationsUnclassified', 'Scheme Operations', 'Scheme Operations', 'Merchant Operations', 'Operations'),
        ('AccountManagementPool', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('AcquiringJapan', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Commercial_Tools_Support', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('FinanceQueue', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Impact', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('InternalSupportCasesQueue', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Lead Qualification', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('LeadQualification', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('LPM', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Nordics_Cases', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('PooledPartnerManagement', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Privacy', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Service_Case', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Risk', 'Risk', 'Risk', 'Professional Services', 'Operations'),
        ('RiskUnclassified', 'Risk', 'Risk', 'Professional Services', 'Operations'),
        ('Warehousing & Logistics', 'Warehousing & Logistics', 'Supply Chain', 'Supply Chain', 'Operations'),
        ('WarehousingLogistics', 'Warehousing & Logistics', 'Supply Chain', 'Supply Chain', 'Operations'),
        ('ComplaintsSupport', 'Operational Support', 'Operational Support', 'Support', 'Operations'),
        ('FranchiseesSupportFLS', 'Operational Support', 'Operational Support', 'Support', 'Operations'),
        ('OperationalSupport', 'Operational Support', 'Operational Support', 'Support', 'Operations'),
        ('PlatformMonitoring', 'Platform Monitoring', 'Platform Operations', 'Support', 'Operations'),
        ('PlatformMonitoringUnclassified', 'Platform Monitoring', 'Platform Operations', 'Support', 'Operations'),
        ('SupportUnclassified', 'Support Unclassified', 'Support Unclassified', 'Support', 'Operations'),
        ('AccountSetupOperationsGeneralists', 'Account Setup Operations', 'Technical Support', 'Support', 'Operations'),
        ('AccountSetupOperationsSpecialists', 'Account Setup Operations', 'Technical Support', 'Support', 'Operations'),
        ('Disputes', 'Disputes', 'Technical Support', 'Support', 'Operations'),
        ('DisputesUnclassified', 'Disputes', 'Technical Support', 'Support', 'Operations'),
        ('PartnerSupport', 'Technical Support', 'Technical Support', 'Support', 'Operations'),
        ('TechnicalSupportGeneralists', 'Technical Support', 'Technical Support', 'Support', 'Operations'),
        ('TechnicalSupportSpecialists', 'Technical Support', 'Technical Support', 'Support', 'Operations'),
        ('Account Management Pool', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Account Setup Operations', 'Account Setup Operations', 'Technical Support', 'Support', 'Operations'),
        ('Acquiring Japan', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Anti Money Laundering (AML)', 'Anti Money Laundering', 'Anti Money Laundering', 'Merchant Operations', 'Operations'),
        ('Claims Support', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Commercial Staging', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Commercial Tools Support', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Complaints Support', 'Operational Support', 'Operational Support', 'Support', 'Operations'),
        ('Credit Risk', 'Credit Risk', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('Customer Risk Review', 'Customer Risk Review', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('Customer Risk Review Unclassified', 'Customer Risk Review', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('Disputes Unclassified', 'Disputes', 'Technical Support', 'Support', 'Operations'),
        ('Finance Queue', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Franchisees Support (FLS)', 'Operational Support', 'Operational Support', 'Support', 'Operations'),
        ('Internal Support Cases Queue', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Lead Qualification Unclassified', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Legal', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Merchant Fraud', 'Merchant Fraud', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('Merchant Fraud Unclassified', 'Merchant Fraud', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('Merchant Potential Liability (MPL)', 'Merchant Potential Liability', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('MPL Unclassified', 'Merchant Potential Liability', 'Merchant Risk', 'Merchant Operations', 'Operations'),
        ('Nordics Cases', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Onboarding Unclassified', 'Onboarding', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('Operational Support', 'Operational Support', 'Operational Support', 'Support', 'Operations'),
        ('Partner Support', 'Technical Support', 'Technical Support', 'Support', 'Operations'),
        ('Payment Card Industry Operations (PCI)', 'Payment Card Industry Operations', 'Scheme Operations', 'Merchant Operations', 'Operations'),
        ('Payment Methods', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('PCI Operations Unclassified', 'Payment Card Industry Operations', 'Scheme Operations', 'Merchant Operations', 'Operations'),
        ('Platform Monitoring', 'Platform Monitoring', 'Platform Operations', 'Support', 'Operations'),
        ('Platform Monitoring Unclassified', 'Platform Monitoring', 'Platform Operations', 'Support', 'Operations'),
        ('Platforms KYC Operations', 'KYC Operations', 'Customer Due Diligence', 'Merchant Operations', 'Operations'),
        ('Pooled Partner Management', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Risk Unclassified', 'Risk', 'Risk', 'Professional Services', 'Operations'),
        ('Scaled Partner Management', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Scheme Operations Unclassified', 'Scheme Operations', 'Scheme Operations', 'Merchant Operations', 'Operations'),
        ('Service Case', 'Not Operations', 'Not Operations', 'Not Operations', 'Not Operations'),
        ('Support Unclassified', 'Support Unclassified', 'Support Unclassified', 'Support', 'Operations'),
        ('Technical Support', 'Technical Support', 'Technical Support', 'Support', 'Operations'),
        ('Underwriting Unclassified', 'Underwriting', 'Customer Due Diligence', 'Merchant Operations', 'Operations')
    ]
    return spark.createDataFrame(data=data, schema=schema)

In [15]:
spark = SparkSession.builder.getOrCreate()
queue_mapping_df = create_queue_mapping_salesforce_df(spark)
queue_mapping_df.show(5)

AML,Anti Money Laundering,Anti Money Laundering,Merchant Operations,Operations
AntiMoneyLaunderingAML,Anti Money Laundering,Anti Money Laundering,Merchant Operations,Operations
CustomerRiskReview,Customer Risk Review,Customer Due Diligence,Merchant Operations,Operations
CustomerRiskReviewUnclassified,Customer Risk Review,Customer Due Diligence,Merchant Operations,Operations
PlatformsKYCOperations,KYC Operations,Customer Due Diligence,Merchant Operations,Operations


In [16]:
queue_mapping_df.write.mode("overwrite").parquet(f"{output_directory}/queue_mapping_df.parquet")

In [17]:
queue_mapping_df = spark.read.parquet(f"{output_directory}/queue_mapping_df.parquet")

### Support Squad logic
https://looker.is.adyen.com/projects/spark/files/support_tooling/views/com_orch_ingestion_salesforce_cases_global.view.lkml  
dimension: support_team 

In [18]:
def add_support_team_column(df):
    
    support_team_col = (
        F.when(
            (F.col('language_c') == 'ja') & 
                ((F.col('team') == 'Operational Support') | (F.col('team') == 'Account Setup Operations'))
            , 'Japan - Operational Support'
        ).when(
            (F.col('language_c') == 'ja') & 
                (F.col('team') == 'Disputes')
            , 'Japan - Disputes'
        ).when(
            (F.col('language_c') == 'ja') & 
                (F.col('team') == 'Technical Support')
            , 'Japan - Technical Support'
        ).when(
            (F.col('language_c') == 'zh'), 'China'
        ).when(
            F.col('team') == 'Account Setup Operations', 'Account Setup Operations'
        ).when(
            (F.col('team') == 'Operational Support') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ), 'Operational Support'
        ).when(
            (F.col('team') == 'Disputes') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ), 'Disputes'
        ).when(
            (F.col('solution_focus_c') == 'Ecom APMs') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'Ecom APM'
        ).when(
            (F.col('solution_focus_c') == 'Ecom Cards') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'Ecom Cards'
        ).when(
            (F.col('solution_focus_c') == 'Ecom Core') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'Ecom Core'
        ).when(
            (F.col('solution_focus_c') == 'Ecom Integration') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'Ecom Integration'
        ).when(
            (F.col('solution_focus_c') == 'Finance') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'Finance'
        ).when(
            (F.col('solution_focus_c') == 'IPP Adv. Payments') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'IPP Adv. Payments'
        ).when(
            (F.col('solution_focus_c') == 'IPP Core') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'IPP Core'
        ).when(
            (F.col('solution_focus_c') == 'IPP Integration') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'IPP Integrations'
        ).when(
            (F.col('solution_focus_c') == 'Platforms') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'Platforms'
        ).when(
            (F.col('solution_focus_c') == 'Partner Support') & (
                (F.col('case_origin_region_c') != 'LATAM') | (F.col('case_origin_region_c').isNull())
            ) & (
                (F.col('language_c') != 'ja') & (F.col('language_c') != 'zh') | (F.col('language_c').isNull())
            ), 'Partner Support'
        ).when(
            (F.col('case_origin_region_c') == 'LATAM') & (
                (F.col('team') == 'Disputes') | 
                (F.col('team') == 'Operational Support')
            ), 'LATAM - Operational Support'
        ).when(
            (F.col('case_origin_region_c') == 'LATAM') & (
                (F.col('team') == 'Technical Support')
            ), 'LATAM - Technical Support'
        ).when(
            (F.col('source_region') == 'india') & (
                (F.col('team') == 'Technical Support')
            ), 'India - Technical Support'
        ).
        otherwise(None)
    )
    
    return df.withColumn(
        'support_team',
        F.when(F.col('domain') == 'Support', support_team_col).otherwise(None)
    )


### Load and Save tables

In [19]:
cases_df = spark.table("com_orch.ingestion_salesforce_cases_global")
cases_df.show(2)

05377912,None,None,2025-08-10 18:19:16,None,0053W000002tc0OQAQ,2025-08-10 18:19:16,2025-08-10,EUR,"Missing: Type, Reason, Associated Account and Contact",60.0,None,False,False,500QD00000e7IMGYA2,Financial Services,True,False,None,0053W000002tc0OQAQ,2025-08-10 18:19:17,None,None,None,0.0,False,Email,00G3W000001WN4SUAW,None,support@adyen.com,00G3W000001WN4SUAW,Medium,None,None,0123W000000L3aoQAC,None,None,02sQD00000XsFK5YAN,Completed,None,None,e23b3d598cff9ed9783ed768614256ee57716d4b7c50b2ffb5dce695df355ff5,None,2025-08-10 18:19:18,None,None,None,None,None,None,None,0012400000UMEDtAAP,None,None,true,OutOfOffice,Support Unclassified,SupportUnclassified,None,Service Case,None,None,None,"<a href=""/00G3W000001WN4S"" target=""_self"">Support Unclassified</a>",None,None,False,None,None,None,SupportUnclassified,false,None,Digital,None,False,None,None,None,None,None,None,None,None,None,False,False,None,2026-05-07 03:13:41.149449,global,False,False,None
05377926,None,None,2025-08-10 18:24:16,None,0053W000002tc0OQAQ,2025-08-10 18:24:16,2025-08-10,EUR,"Missing: Type, Reason, Associated Account and Contact",60.0,None,False,False,500QD00000e7IMHYA2,Financial Services,True,False,None,0053W000002tc0OQAQ,2025-08-10 18:24:17,None,None,None,0.0,False,Email,00G3W000001WN4SUAW,None,support@adyen.com,00G3W000001WN4SUAW,Medium,None,None,0123W000000L3aoQAC,None,None,02sQD00000XsFK6YAN,Completed,None,None,e23b3d598cff9ed9783ed768614256ee57716d4b7c50b2ffb5dce695df355ff5,None,2025-08-10 18:24:18,None,None,None,None,None,None,None,0012400000UMEDtAAP,None,None,true,OutOfOffice,Support Unclassified,SupportUnclassified,None,Service Case,None,None,None,"<a href=""/00G3W000001WN4S"" target=""_self"">Support Unclassified</a>",None,None,False,None,None,None,SupportUnclassified,false,None,Digital,None,False,None,None,None,None,None,None,None,None,None,False,False,None,2026-05-07 03:13:41.149449,global,False,False,None


In [20]:
cases_df.write.mode("overwrite").parquet(f"{output_directory}/cases_df.parquet")

In [21]:
cases_df = spark.read.parquet(f"{output_directory}/cases_df.parquet")

In [22]:
measures_df = spark.table("com_orch.salesforcecasescalculatedmeasures")
measures_df.show(2)

None,5003W00000PPkEbQAL,None,None,None,None,None,None,None,None,None,None,0,0,False,0,None,0,None,None,0,0,global
1,5003W00000PSp4oQAD,None,None,None,5,None,None,None,143,None,None,0,2689,False,168,None,0,None,None,0,0,global


In [23]:
measures_df.write.mode("overwrite").parquet(f"{output_directory}/measures_df.parquet")

In [24]:
measures_df = spark.read.parquet(f"{output_directory}/measures_df.parquet")

### Combine data

In [25]:
def process_cases(cases_df, queue_mapping_df, measures_df):
    """
    Processes the cases DataFrame by:
    1. Joining with queue mapping data
    2. Joining with measures data
    3. Adding support team column
    4. Formatting the month column
    5. Creating a category column based on emails_sent
    6. Filtering out unwanted records
    7. Aggregating to count unique IDs by month, domain, subdomain, team, support_team, origin, and category
    
    Args:
        cases_df (DataFrame): The main cases DataFrame.
        queue_mapping_df (DataFrame): The mapping DataFrame for queues.
        measures_df (DataFrame): The measures DataFrame.

    Returns:
        DataFrame: Aggregated DataFrame with unique_id_count per group.
    """
    
    # Step 1: Join with queue mapping
    clean_cases = cases_df.join(queue_mapping_df, cases_df.previous_queue_c == queue_mapping_df.queue, how='left') \
                          .drop(queue_mapping_df.queue)

    # Step 2: Join with measures data
    final_cases = clean_cases.join(measures_df, clean_cases.id == measures_df.case_id, how='left') \
                             .drop(measures_df.case_id) \
                             .drop(measures_df.source_region)   

    # Step 3: Add support team column (assuming the function exists)
    final_cases = add_support_team_column(final_cases)

    # Step 4: Format 'month' column
    final_cases = final_cases.withColumn('month', F.date_format(F.date_trunc('MM', F.col('created_date')), 'yyyy-MM-dd'))

    # Step 5: Create 'category' column
        # https://looker.is.adyen.com/projects/spark/files/support_tooling/views/com_orch_ingestion_salesforce_cases_global.view.lkml
        # dimension: case_category
    final_cases = final_cases.withColumn(
        'category', 
        F.when(F.col('emails_sent') > 0, 'Service').otherwise('Internal')
    )

    # Step 6: Apply filtering conditions
        # https://looker.is.adyen.com/projects/spark/files/support_tooling/explores/cases_main.explore.lkml
        # replicate sql_always_where
    filtered_cases = final_cases.filter(
        (~F.col("is_deleted") | F.col("is_deleted").isNull()) &  
        (~(F.col("status") == "Merged") | F.col("status").isNull()) &  
        (~F.col("auto_close_reason_c").isin(["Blocked Email", "Shopper Case", "Zendesk Transfer", "Invalid Case", "Case Automation", "OutOfOffice", "RedirectedToCustomerArea", "Undelivered","Complaints Auto-Reply"]) | 
         F.col("auto_close_reason_c").isNull())  
    )

    # Step 7: Aggregate unique ID count
    aggregated_final_cases = filtered_cases.groupBy(
        'month', 'domain', 'subdomain', 'team', 'support_team', 'origin', 'category'
    ).agg(
        F.countDistinct('id').alias('unique_id_count')
    )

    return aggregated_final_cases

In [26]:
aggregated_cases_df = process_cases(cases_df, queue_mapping_df, measures_df)
aggregated_cases_df.show(5)

2024-02-01,Support,Technical Support,Technical Support,Ecom Cards,Email,Service,937
2024-09-01,Support,Operational Support,Operational Support,Operational Support,Email,Internal,990
2025-01-01,Support,Technical Support,Account Setup Operations,Account Setup Operations,Webform,Service,393
2025-10-01,Support,Technical Support,Technical Support,LATAM - Technical Support,Email,Service,588
2025-11-01,Support,Operational Support,Operational Support,Operational Support,Email,Service,4350


In [29]:
aggregated_cases_df.select("support_team").distinct().show()

Account Setup Operations
Partner Support
None
IPP Integrations
IPP Core
Finance
Japan - Operational Support
China
LATAM - Operational Support
Japan - Technical Support
Disputes


#### Validation of table above
https://looker.is.adyen.com/explore/spark_support_tooling/cases?toggle=fil&qid=Ob2fOb1wbK5crp8W1cenhL

In [27]:
aggregated_cases_df.write.mode("overwrite").parquet(f"{output_directory}/aggregated_cases_df.parquet")

In [28]:
aggregated_cases_df = spark.read.parquet(f"{output_directory}/aggregated_cases_df.parquet")

## Backoffice data

### Load tables

In [34]:
account_creation_df = spark.table("pspconfig.account")
account_creation_df.show(2)

1,Root,None,1,1,2008-02-19 10:10:29.094096,None,Europe/Amsterdam,None,None
2,AdyenPspService,The Adyen PSP Account,2,1,2008-02-19 10:10:29.094096,Active,Europe/Amsterdam,1,None


In [35]:
account_closure_df = spark.table("pspconfig.accountclosure")
account_closure_df.show(2)

106200,Shuttle migration to SumUp. Closing ALL contacted Shuttle merchants. Contact Tobias Uebbing if there are any questions.,2017-08-30 00:00:00,2017-09-01 00:00:00,2017-12-01 09:37:14.335235,APPROVED,Automatic,False,Other
231069,End,2017-05-02 00:00:00,2017-06-01 00:00:00,2018-01-01 01:11:50.352421,APPROVED,Automatic,False,Other


In [36]:
account_country_df = spark.table("pspconfig.accountdata")
account_country_df.show(2)

8891545,767,(default)
8891548,767,(default)


In [37]:
company_account_df = spark.table("revenue_operations.company_account")
company_account_df.show(2)

EuroMailBV,9196,None,None,Closed,None,EuroMailBV,Managed Account,None,None,None,None,None,None,None,None,None,None,NL,EMEA,Amsterdam,Netherlands,Dedicated,patrickb,None,None,roelant,None,None,None,None,None,None,False,None,2012-04-11,2012-02-28,None,None,None
Benetton,12214,None,None,Closed,None,Benetton,Managed Account,None,None,None,None,None,None,None,None,None,None,IT,NA,Toronto,Canada,Dedicated,vincentt,None,None,volker,None,None,None,None,None,None,False,None,2012-09-14,2012-08-31,None,None,None


In [38]:
region_map_df = spark.table("central_analytics.countryregionmapping")
region_map_df.show(2)

LATAM,BVT,BV,Bouvet Island,South America,005,ISO 3166-2:BV,074,019,419,Americas,Latin America and the Caribbean
LATAM,BRA,BR,Brazil,South America,005,ISO 3166-2:BR,076,019,419,Americas,Latin America and the Caribbean


In [39]:
def process_accounts(account_creation_df,account_closure_df,account_country_df):
    
    combined_data = (
        account_creation_df
        .join(account_closure_df, (account_creation_df.accountid == account_closure_df.accountid) & (account_creation_df.accounttypeid == 7), how='left')
        .join(account_country_df, (account_creation_df.accountid == account_country_df.accountid) & (account_country_df.accountdatatypeid == 4), how='inner')
    ) 
    
    adyen_parent_id = account_creation_df.filter(F.col("accountcode") == "AdyenPspService").select("accountid").first()
    adyen_parent_id_value = adyen_parent_id.accountid if adyen_parent_id else None

    clean_accounts = combined_data.filter(
        (F.col("accounttypeid") == 7)  &
        (F.col("parentid") == F.lit(adyen_parent_id_value))
    ).select(
        F.col("accountcode").alias("account_code"),
        F.col("creationdate").alias("creation_date"),
        F.col("closingdate").alias("closing_date"),
        F.col("data1").alias("country_code")
    )
    
    return clean_accounts

In [40]:
account_details_df = process_accounts(account_creation_df,account_closure_df,account_country_df)
account_details_df.show(5)

StudieBoekenCom,2008-09-29 15:36:51.604639,2017-06-01 01:09:38.757481,NL
WritingTravellers,2009-08-05 11:35:48.813612,2023-12-01 05:38:14.216044,NL
HouseTrip,2009-09-04 15:08:39.948516,2020-07-01 01:10:30.592209,CH
Cowite,2009-09-14 17:32:26.843247,None,NL
SharingBase,2010-11-24 12:02:48.977856,2021-08-01 01:15:13.903746,NL


In [41]:
account_details_df.write.mode("overwrite").parquet(f"{output_directory}/account_details_df.parquet")

In [42]:
account_details_df = spark.read.parquet(f"{output_directory}/account_details_df.parquet")

In [43]:
def active_accounts(account_details_df,company_account_df,region_map_df):
    
    combined_data = (
        account_details_df
        .join(region_map_df, on='country_code', how='left')
        .join(company_account_df, (account_details_df.account_code ==company_account_df.company_account_code), how='left')
    ) \
    .withColumn("pillar_clean", F.coalesce(F.col("pillar"), F.col("pillar_bmboad"), F.col("pillar_cbm"))) \
    .withColumn("closing_date", 
                F.when(F.col("closing_date").isNull(), F.current_date())
                .otherwise(F.col("closing_date"))
               ) \
    .filter(
        (~F.col("creation_date").isNull())
    ) \
    .select(
        F.col("adyen_region").alias("region"),
        F.col("managed_from_region").alias("managed_region"),
        F.col("account_code"),
        F.col("creation_date"),
        F.col("closing_date"),
        F.col("first_live_transaction_date"),
        F.col("pillar_clean")
    )
    
    month_data = combined_data.withColumn(
        "month_seq", F.explode(F.sequence(
            F.date_trunc("month", F.col("creation_date")),
            F.date_trunc("month", F.col("closing_date")),
            F.expr("INTERVAL 1 MONTH")
        ))
    )
    
    clean_data = month_data.withColumn(
        "active",
        F.when(
            (F.col("first_live_transaction_date").isNotNull()) &
            (F.col("month_seq") >= F.date_trunc("month", F.col("first_live_transaction_date"))) &
            (F.col("month_seq") <= F.date_trunc("month", F.col("closing_date"))),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    
    clean_data = clean_data.withColumn("creation_date", F.date_format(F.date_trunc('MM', F.col('creation_date')), 'yyyy-MM-dd'))
    clean_data = clean_data.withColumn("closing_date", F.date_format(F.date_trunc('MM', F.col('closing_date')), 'yyyy-MM-dd'))
    clean_data = clean_data.withColumn("month_seq", F.date_format(F.date_trunc('MM', F.col('month_seq')), 'yyyy-MM-dd'))
    
    return clean_data
    

In [44]:
active_accounts_df = active_accounts(account_details_df,company_account_df,region_map_df)
active_accounts_df.show(5)

APAC,APAC,37GAMES,2022-08-01,2026-02-01,2022-08-29,Digital,2022-08-01,1
APAC,APAC,37GAMES,2022-08-01,2026-02-01,2022-08-29,Digital,2022-09-01,1
APAC,APAC,37GAMES,2022-08-01,2026-02-01,2022-08-29,Digital,2022-10-01,1
APAC,APAC,37GAMES,2022-08-01,2026-02-01,2022-08-29,Digital,2022-11-01,1
APAC,APAC,37GAMES,2022-08-01,2026-02-01,2022-08-29,Digital,2022-12-01,1


In [45]:
active_accounts_df.write.mode("overwrite").parquet(f"{output_directory}/active_accounts_df.parquet")

In [46]:
active_accounts_df = spark.read.parquet(f"{output_directory}/active_accounts_df.parquet")

In [47]:
active_accounts_df.select('active').distinct().show()

1
0


In [48]:
def active_account_summary(active_accounts_df):

    active_accounts_df = active_accounts_df.filter(active_accounts_df['active'] == 1)
    
    aggregated_accounts = active_accounts_df.groupBy(
        'month_seq', 'region', 'managed_region', 'pillar_clean'
    ).agg(
        F.countDistinct('account_code').alias('unique_accounts')
    )
    
    return aggregated_accounts

In [49]:
active_account_summary_df = active_account_summary(active_accounts_df)
active_account_summary_df.show(5)

2016-10-01,EMEA,EMEA,Digital,321
2022-01-01,EMEA,EMEA,Platforms,84
2013-11-01,EMEA,None,Digital,382
2024-07-01,NA,None,UC,109
2015-03-01,EMEA,None,UC,164


In [50]:
active_account_summary_df.write.mode("overwrite").parquet(f"{output_directory}/active_account_summary_df.parquet")

In [51]:
active_account_summary_df = spark.read.parquet(f"{output_directory}/active_account_summary_df.parquet")

In [52]:
grouped = active_account_summary_df.groupBy('month_seq').agg(F.sum('unique_accounts').alias('total_unique_accounts'))
grouped.orderBy('month_seq', ascending=False).show(10)

2026-02-01,9782
2026-01-01,9750
2025-12-01,9728
2025-11-01,9676
2025-10-01,9609
2025-09-01,9568
2025-08-01,9498
2025-07-01,9464
2025-06-01,9407
2025-05-01,9351


## Aircall data

### Load tables

In [12]:
call_history_df = spark.read.csv(f"{output_directory}/Aircall_Combined_Detailed.csv",header=True, inferSchema=True)

In [13]:
call_history_df.show(5)

2023-12-31,[Service] Sweden (46844686719),Inbound - Out Of Opening Hours,358405702602,46844686719
2023-12-31,[Service] London (442045799461),Inbound - Out Of Opening Hours,358405702602,442045799461
2023-12-31,[Service] Netherlands (31208080398),Inbound - Answered,31611295403,31208080398
2023-12-31,[Service] Netherlands (31208080398),Inbound - Answered,31631767677,31208080398
2023-12-31,[Service] London (442045799461),Inbound - Answered,447423511168,442045799461


In [55]:
def config_calls(df):
    
    df = df.withColumn('month', F.date_format(F.date_trunc('MM', F.col('Call Created Date')), 'yyyy-MM-dd'))
    
    df = df.withColumn(
        'support_team',
        F.when(
            (F.col('Aircall number').contains('Japan')) , 'Japan'
        ).when(
            (F.col('Aircall number').contains('Brazil')) , 'LATAM'
        ).when(
            (F.col('Aircall number').contains('BR Duty')) , 'LATAM'
        ).otherwise("Operational Support") 
    )
    
    filtered_df = df.filter(
        (F.col('Aircall number').startswith('[Service]')) &
        (F.col('Aircall number') != '[Service]  POS 2ndLine') &
        (F.col('Call direction - type').isin('Inbound - Answered', 'Outbound'))
    )
    
    result_df = filtered_df.groupBy('month', 'support_team') \
        .agg(F.count('*').alias('calls')) \
        .orderBy('month', 'support_team')
    
    return result_df

In [56]:
clean_calls_df = config_calls(call_history_df)
clean_calls_df.show(100)

2023-01-01,Japan,67
2023-01-01,LATAM,54
2023-01-01,Operational Support,954
2023-02-01,Japan,77
2023-02-01,LATAM,64
2023-02-01,Operational Support,1034
2023-03-01,Japan,121
2023-03-01,LATAM,50
2023-03-01,Operational Support,1306
2023-04-01,Japan,77
2023-04-01,LATAM,49


In [57]:
clean_calls_df.write.mode("overwrite").parquet(f"{output_directory}/clean_calls_df.parquet")

## Aircall data WEIRD FORMAT

In [19]:
calls_1 = spark.read.csv(f"{output_directory}/181797_part_aa.csv",header=True, inferSchema=True)

In [20]:
calls_2 = spark.read.csv(f"{output_directory}/181797_part_ab.csv",header=False, inferSchema=True)

In [21]:
calls_1.show(2)

CA6a17d58d93d20d887253c53f1cf0046e,1733413946,inbound,755,778,41787331001,41315500977,467579,[Service] Switzerland (German) (41315500977),[Service] Switzerland (German),181797,1128153,Tatiana A.,2023-12-11 15:27:08,2023-12-11 15:27:31,2023-12-11 15:40:06,Switzerland,geographic_local,0.0,None,None,None,Yes,No,None,0,Not available,tatiana.aguilar@adyen.com,2023-12-11,2026-03-23 15:27:08,Europe/Berlin,CH,APP,None,None,None
CAffd614808e920c0a99ebcf5692288812,1987770746,inbound,None,5,anonymous,31858888138,62512,NETHERLANDS (Tech Support - Outbound) (31858888138),NETHERLANDS (Tech Support - Outbound),181797,None,None,2024-04-30 12:24:20,None,2024-04-30 12:24:26,None,national,0.0,None,None,out_of_opening_hours,No,Yes,None,None,Not available,None,2024-04-30,2026-03-23 12:24:20,Europe/Paris,NL,None,None,None,None


In [22]:
calls_2.show(2)

CA957e402f665cadffec07dbbc0bf72e4d,65808378,inbound,96,105,17315550150,4932221098390,63516,GERMANY (4932221098390),GERMANY,181797,None,Philipp Hackel,2018-07-18 15:54:14,2018-07-18 15:54:23,2018-07-18 15:55:59,United States of America,None,0.0,None,None,None,Yes,No,None,0,Not available,1550589188+philipp.hackel@adyen.com,2018-07-18,2026-03-23 15:54:14,Europe/Paris,DE,APP,None,None,None
CA2b4bed162886b3579921555a54fcadf8,86907312,inbound,87,101,13012347151,14153671502,63044,US (14153671502),US,181797,None,Tara Fokker,2018-11-21 18:29:11,2018-11-21 18:29:26,2018-11-21 18:30:53,United States of America,None,0.0,None,None,None,Yes,No,None,0,Not available,1556548126+tara.fokker@adyen.com,2018-11-21,2026-03-23 18:29:11,Europe/Paris,US,APP,None,None,None


In [17]:
calls_1.select('Calls Missed Reason').distinct().show()

agents_did_not_answer
None
short_abandoned
out_of_opening_hours
abandoned_in_classic
abandoned_in_ivr
no_available_agent


In [26]:
new_calls = calls_1.union(calls_2)

In [31]:
def new_config_calls(df):
    
    df = df.withColumn('month', F.date_format(F.date_trunc('MM', F.col('Calls Created Time')), 'yyyy-MM-dd'))
    
    df = df.withColumn(
        'support_team',
        F.when(
            (F.col('Numbers Number Name').contains('Japan')) , 'Japan'
        ).when(
            (F.col('Numbers Number Name').contains('Brazil')) , 'LATAM'
        ).when(
            (F.col('Numbers Number Name').contains('BR Duty')) , 'LATAM'
        ).otherwise("Operational Support") 
    )
    
    filtered_df = df.filter(
        (F.col('Numbers Number Name').startswith('[Service]')) &
        (F.col('Numbers Number Name') != '[Service]  POS 2ndLine') &
        (F.col('Calls Direction').isin('inbound', 'outbound')) &
        (F.col('Calls Missed Reason').isNull() | (F.col('Calls Missed Reason') == "")) &
        (F.year(F.col('Calls Created Time')) >= 2023)
    )
    
    result_df = filtered_df.groupBy('month', 'support_team') \
        .agg(F.count('*').alias('calls')) \
        .orderBy('month', 'support_team')
    
    return result_df

In [34]:
clean_calls_v2_df = new_config_calls(new_calls)
clean_calls_v2_df.show(10)

2023-01-01,Japan,157
2023-01-01,LATAM,100
2023-01-01,Operational Support,1137
2023-02-01,Japan,167
2023-02-01,LATAM,107
2023-02-01,Operational Support,1257
2023-03-01,Japan,249
2023-03-01,LATAM,99
2023-03-01,Operational Support,1511
2023-04-01,Japan,175


In [35]:
clean_calls_v2_df.write.mode("overwrite").parquet(f"{output_directory}/clean_calls_df.parquet")

## Aircall data BDP

above code needs to be deleted once data in the BDP is older than 6 months

### Load tables

In [30]:
calls_df = spark.table("com_orch.ingestion_aircall_calls_pii")
calls_df.show(2)

3740478315,CAf39a1bb97e4b3a9310c0e80f9f5fd14a,2026-05-05 02:36:09,2026-05-05 02:36:56,2026-05-05 02:42:25,376,done,inbound,False,None,467571,693729,"{""company_name"": ""Tiffanys"", ""created_at"": 1730230662, ""direct_link"": ""https://api.aircall.io/v1/contacts/224650877"", ""emails"": [], ""first_name"": """", ""id"": 224650877, ""information"": ""TiffanyAndCoAustraliaPtyLtdECOM"", ""is_shared"": true, ""last_name"": """", ""phone_numbers"": [{""id"": 531483301, ""label"": ""Main"", ""new_contacts_service_id"": null, ""value"": ""+61413683266""}], ""updated_at"": 1754034227}",None,None,None,+61 413 683 266,"[{""content"": ""Case: 07773610\nCaller name: Tessa Lin \nCompany/Merchant account: Tiffanys \nEmail : tessa.lin@tiffany.com \nIssue: (1) Need to create 2 new merchant account, one for store and another for PBL. Advise that she will need company level access as well as create new merchant account role which she is missing. Will send her the guide so that she can reach out internally. (2) After recent system upgrade particular store ID is missing in Aggregate settlement report, request for them to provide the report to further investigate."", ""id"": 126581716, ""posted_at"": 1777942606, ""posted_by"": {""availability_status"": ""unavailable"", ""available"": false, ""created_at"": ""2021-12-02T01:40:08.000Z"", ""direct_link"": ""https://api.aircall.io/v1/users/693729"", ""email"": ""cheryl.hau@adyen.com"", ""extension"": ""059"", ""id"": 693729, ""language"": ""en-US"", ""name"": ""Cheryl H."", ""state"": ""always_closed"", ""time_zone"": ""Asia/Singapore"", ""wrap_up_time"": 300}}]",[],"[{""branch"": ""Consent given"", ""created_at"": ""2026-05-05T00:36:49.768Z"", ""id"": ""84925151-c5f8-4ed9-99fd-91b8c0345119"", ""key"": ""1"", ""title"": ""Intro + consent to record?"", ""transition_ended_at"": ""2026-05-05T00:36:49.750Z"", ""transition_started_at"": ""2026-05-05T00:36:09.136Z""}]",2026-05-06 05:21:04.223786
3740587774,CA573e627cbee064bea199ae917b4aa834,2026-05-05 04:05:25,2026-05-05 04:06:21,2026-05-05 04:11:29,364,done,inbound,False,None,467571,1594692,None,None,None,None,+61 407 218 212,"[{""content"": ""The merchant wanted information about the MPL amount that is with held for the merchant account PetalsNetworkPtyLtd."", ""id"": 126587010, ""posted_at"": 1777947549, ""posted_by"": {""availability_status"": ""available"", ""available"": true, ""created_at"": ""2025-04-30T07:48:09.000Z"", ""direct_link"": ""https://api.aircall.io/v1/users/1594692"", ""email"": ""rakesh.k@adyen.com"", ""extension"": ""088"", ""id"": 1594692, ""language"": ""en-US"", ""name"": ""Rakesh K"", ""state"": ""always_opened"", ""time_zone"": ""Asia/Kolkata"", ""wrap_up_time"": 300}}]",[],"[{""branch"": ""Consent given"", ""created_at"": ""2026-05-05T02:06:07.586Z"", ""id"": ""84925151-c5f8-4ed9-99fd-91b8c0345119"", ""key"": ""1"", ""title"": ""Intro + consent to record?"", ""transition_ended_at"": ""2026-05-05T02:06:07.565Z"", ""transition_started_at"": ""2026-05-05T02:05:25.791Z""}]",2026-05-06 05:21:04.223786


https://looker.is.adyen.com/projects/spark/files/support_tooling/views/dt_aircall_number_mapping.view.lkml

In [31]:
def create_numbers_mapping(spark: SparkSession):
    schema = T.StructType([
        T.StructField("number", T.StringType(), nullable=False),
        T.StructField("name", T.StringType(), nullable=False),
        T.StructField("language", T.StringType(), nullable=False),
        T.StructField("timezone", T.StringType(), nullable=False),
        T.StructField("country", T.StringType(), nullable=False),
        T.StructField("service_number", T.StringType(), nullable=False)
    ])
    
    data = [
        ('915244', 'Seleena Grewal', 'en-US', 'Etc/UTC', 'GB', 'No'),
            ('922734', 'Melanie Vecchio', 'en-US', 'Europe/Paris', 'FR', 'No'),
            ('721818', 'Gaspard Le Poittevin - France', 'fr-FR', 'Etc/UTC', 'FR', 'No'),
            ('722709', 'Jake Guenoun - United States', 'en-US', 'America/Chicago', 'US', 'No'),
            ('726382', 'Felix Lantz - Switzerland Mobile', 'en-US', 'Europe/Stockholm', 'SE', 'No'),
            ('710434', 'Netherlands mobile 2', 'en-US', 'Etc/UTC', 'NL', 'No'),
            ('661597', 'Australia - Innes Galloway', 'en-US', 'Etc/UTC', 'AU', 'No'),
            ('778488', 'Simon Lorenz - Germany', 'de-DE', 'Europe/Berlin', 'DE', 'No'),
            ('778500', 'Heorchi Talochka - Germany', 'de-DE', 'Europe/Berlin', 'DE', 'No'),
            ('760644', 'External US number #2', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('760646', 'External US number #3', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('802780', 'Lukas Breitmeier - Germany', 'de-DE', 'Europe/Berlin', 'DE', 'No'),
            ('768607', 'Sean Cryer - Canada', 'en-US', 'America/Chicago', 'CA', 'No'),
            ('897380', 'New Number', 'en-US', 'Etc/UTC', 'SG', 'No'),
            ('927641', '15084043391', 'en-US', 'America/Chicago', 'US', 'No'),
            ('833182', 'Tommaso Caccia', 'en-US', 'Etc/UTC', 'IT', 'No'),
            ('928241', 'Jeongmi Kim', 'en-US', 'Asia/Tokyo', 'JP', 'No'),
            ('928244', 'Carolina mota', 'en-US', 'America/Sao_Paulo', 'BR', 'No'),
            ('930471', 'New 415 SFO Number', 'en-US', 'America/Chicago', 'US', 'No'),
            ('931220', 'Tasmiyah Chowdhury', 'en-US', 'America/Chicago', 'US', 'No'),
            ('661564', 'United Kingdom - Hasan Zaman', 'en-US', 'Etc/UTC', 'GB', 'No'),
            ('949139', 'number 6', 'en-US', 'Asia/Tokyo', 'JP', 'No'),
            ('949140', 'number 7', 'en-US', 'Asia/Tokyo', 'JP', 'No'),
            ('949142', 'number 9', 'en-US', 'Asia/Tokyo', 'JP', 'No'),
            ('949143', 'number 10', 'en-US', 'Asia/Tokyo', 'JP', 'No'),
            ('949144', 'number 11', 'en-US', 'Asia/Tokyo', 'JP', 'No'),
            ('661569', 'United States - Thomas Jeremy Malato (Prev: TJ Jude)', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('496801', 'Underwriting APAC', 'en-US', 'Asia/Singapore', 'SG', 'No'),
            ('779831', 'Gonzalo Pinto - Netherlands', 'en-US', 'Europe/Amsterdam', 'NL', 'No'),
            ('969629', 'Ariana Saghafi', 'en-US', 'America/Los_Angeles', 'US', 'No'),
            ('877253', 'Landline - France', 'fr-FR', 'Europe/Paris', 'FR', 'No'),
            ('719923', 'France - non-mobile areacode', 'fr-FR', 'Europe/Paris', 'FR', 'No'),
            ('875344', 'Antoine Laguilliez - Mobile', 'en-US', 'Etc/UTC', 'FR', 'No'),
            ('976677', 'Monique - SDR NYC phone', 'en-US', 'America/Chicago', 'US', 'No'),
            ('62512', 'NETHERLANDS (Tech Support - Outbound)', 'en-US', 'Europe/Paris', 'NL', 'No'),
            ('658866', 'Marina Schwartzman - Brazil', 'en-US', 'Etc/UTC', 'BR', 'No'),
            ('662906', 'France - Irene Giorgio', 'fr-FR', 'Europe/Paris', 'FR', 'No'),
            ('659408', 'Netherlands - Sebastiaan Ferman', 'en-US', 'Etc/UTC', 'NL', 'No'),
            ('660828', 'Netherlands - Thomas Malato', 'en-US', 'Etc/UTC', 'NL', 'No'),
            ('661547', 'Poland - Jakub Laska', 'en-US', 'Etc/UTC', 'PL', 'No'),
            ('661567', 'United States - Brandon Mackles', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('665378', 'United States - August Brown', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('989772', 'Gino Castillo', 'en-US', 'America/Los_Angeles', 'US', 'No'),
            ('710438', 'Grace Breen', 'en-US', 'Etc/UTC', 'GB', 'No'),
            ('1019425', 'Number 02', 'es-ES', 'Europe/Madrid', 'ES', 'No'),
            ('1019426', 'Number 03', 'es-ES', 'Europe/Madrid', 'ES', 'No'),
            ('1019887', '5.26E+11', 'es-ES', 'America/Mexico_City', 'MX', 'No'),
            ('661593', 'United States - Stephen Lee', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('1019424', 'Geovane Santos - Spain Mobile', 'es-ES', 'Europe/Madrid', 'ES', 'No'),
            ('65141', '[Service]  POS 2ndLine', 'en-US', 'Europe/Paris', 'NL', 'No'),
            ('762498', 'Geovane Santos - Brazil', 'en-US', 'Etc/UTC', 'BR', 'No'),
            ('661565', 'United Kingdom - Carrie Cheung', 'en-US', 'Etc/UTC', 'GB', 'No'),
            ('661598', 'SF Gaspard Le Poittevin', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('1050752', 'Vinicius Reis - Brazil ( Personal External)', 'pt-PT', 'America/Sao_Paulo', 'BR', 'No'),
            ('702045', 'Oscar Scoppa - Netherlands', 'en-US', 'Etc/UTC', 'NL', 'No'),
            ('1056687', 'Lili Garcia - EXTERNAL', 'es-ES', 'America/Mexico_City', 'MX', 'No'),
            ('811375', 'Ben Bogard External (03.05.)', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('71101', 'Tech 2ndLine', 'en-US', 'Europe/Paris', 'NL', 'No'),
            ('659403', 'Netherlands - Nick Dennett', 'en-US', 'Etc/UTC', 'NL', 'No'),
            ('775054', 'Jake Parrish - United States', 'en-US', 'America/New_York', 'US', 'No'),
            ('661596', 'France  - Yoann Pouliquen', 'en-US', 'Etc/UTC', 'FR', 'No'),
            ('928242', 'Steven Ogata', 'en-US', 'Asia/Tokyo', 'JP', 'No'),
            ('1039441', 'Filip Vanik - Czechia', 'en-US', 'Europe/Prague', 'CZ', 'No'),
            ('1090565', 'Spain mobile ', 'es-ES', 'Europe/Madrid', 'ES', 'No'),
            ('928243', 'Netherlands 1', 'en-US', 'Europe/Amsterdam', 'NL', 'No'),
            ('1104308', 'IPP TS Outbound ONLY', 'en-US', 'Europe/Amsterdam', 'NL', 'No'),
            ('949135', 'number 1', 'en-US', 'Asia/Tokyo', 'JP', 'No'),
            ('661572', 'United States - Manas Belliappa', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('1135012', 'Samantha Brzozowicz SDR', 'en-US', 'America/Toronto', 'CA', 'No'),
            ('661568', 'United States - Samantha Brzozowicz', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('533920', 'Salesforce test phone number [Service]', 'en-US', 'Europe/Amsterdam', 'NL', 'No'),
            ('577537', '[Service] Franchisee Support: Line-X', 'en-US', 'America/Chicago', 'US', 'Yes'),
            ('662907', 'Camille Jarry', 'fr-FR', 'Europe/Paris', 'FR', 'No'),
            ('661579', 'Guri Grewal', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('234048', '[Service] BR Duty number', 'pt-PT', 'America/Sao_Paulo', 'BR', 'Yes'),
            ('658864', 'BR Black Friday (TKF Merchants)', 'en-US', 'America/Sao_Paulo', 'BR', 'No'),
            ('397755', '[Service] 0800 Brazil Ombudsman', 'en-US', 'America/Sao_Paulo', 'BR', 'Yes'),
            ('467599', '[Service] Belgium', 'en-US', 'Europe/Amsterdam', 'BE', 'Yes'),
            ('467608', '[Service] Czech Republic', 'en-US', 'Europe/Amsterdam', 'CZ', 'Yes'),
            ('222499', '[Service] France L1', 'fr-FR', 'Europe/Paris', 'FR', 'Yes'),
            ('223458', '[Service] Germany L1', 'en-US', 'Europe/Berlin', 'DE', 'Yes'),
            ('468076', '[Service] Hong Kong', 'en-US', 'Europe/Paris', 'HK', 'Yes'),
            ('468144', '[Service] Netherlands', 'en-US', 'Europe/Paris', 'NL', 'Yes'),
            ('222507', '[Service] Netherlands L1', 'en-US', 'Europe/Amsterdam', 'NL', 'Yes'),
            ('467606', '[Service] New Zealand', 'en-US', 'Pacific/Auckland', 'NZ', 'Yes'),
            ('467614', '[Service] Poland', 'en-US', 'Europe/Madrid', 'PL', 'Yes'),
            ('467585', '[Service] Singapore', 'en-US', 'Asia/Singapore', 'SG', 'Yes'),
            ('222508', '[Service] Singapore L1', 'en-US', 'Asia/Singapore', 'SG', 'Yes'),
            ('467566', '[Service] Spain', 'en-US', 'Europe/Paris', 'ES', 'Yes'),
            ('222509', '[Service] Spain L1', 'es-ES', 'Europe/Madrid', 'ES', 'Yes'),
            ('222511', '[Service] UK L1', 'en-US', 'Europe/London', 'GB', 'Yes'),
            ('461604', '[Service] USA - NY', 'en-US', 'America/New_York', 'US', 'Yes'),
            ('467579', '[Service] Switzerland (German)', 'de-DE', 'Europe/Berlin', 'CH', 'Yes'),
            ('420936', '[Service] Japan', 'en-GB', 'Asia/Tokyo', 'JP', 'Yes'),
            ('87820', '[Service] Japan L1 (Exclusive)', 'en-US', 'Asia/Tokyo', 'JP', 'Yes'),
            ('1033687', 'EFP Sub-Merchant (UK)', 'en-US', 'Europe/London', 'GB', 'No'),
            ('1033698', 'EFP Sub-Merchant (US)', 'en-US', 'America/Los_Angeles', 'US', 'No'),
            ('661393', '❗ [Service] Unrecorded Line - First Line Support', 'en-US', 'Europe/Amsterdam', 'NL', 'Yes'),
            ('776408', '[Service] Unrecorded line - General Support', 'en-US', 'Europe/Amsterdam', 'NL', 'Yes'),
            ('467571', '[Service] Australia', 'en-US', 'Europe/Amsterdam', 'AU', 'Yes'),
            ('1067059', 'Nathan Passos External number', 'pt-PT', 'America/Sao_Paulo', 'BR', 'No'),
            ('840913', 'French mobile number 2', 'en-US', 'Etc/UTC', 'FR', 'No'),
            ('65077', 'L1 BR Richemont-Closed', 'pt-BR', 'America/Sao_Paulo', 'BR', 'No'),
            ('910875', 'New Swedish number', 'en-US', 'Etc/UTC', 'SE', 'No'),
            ('915246', 'Landline - Hasan Zaman', 'en-US', 'Etc/UTC', 'GB', 'No'),
            ('913179', 'ACH Phone Number', 'en-US', 'America/Chicago', 'US', 'No'),
            ('925575', 'Nour Sadek', 'en-US', 'Europe/Amsterdam', 'NL', 'No'),
            ('687620', 'Gonzalo - Netherlands', 'en-US', 'Etc/UTC', 'NL', 'No'),
            ('661530', 'Netherlands - Jacopo Ficarelli', 'en-US', 'Etc/UTC', 'NL', 'No'),
            ('710422', 'UK mobile ', 'en-GB', 'Etc/UTC', 'GB', 'No'),
            ('710436', 'Netherlands mobile 3', 'en-US', 'Etc/UTC', 'NL', 'No'),
            ('719922', 'Marie de Woot - France', 'fr-FR', 'Europe/Paris', 'FR', 'No'),
            ('778489', 'Simon Ellerkamp - Germany', 'de-DE', 'Europe/Berlin', 'DE', 'No'),
            ('778493', 'Lina Weisenberg - Germany', 'de-DE', 'Europe/Berlin', 'DE', 'No'),
            ('811371', 'Eric Sisan External (03.05.)', 'en-US', 'Etc/UTC', 'AE', 'No'),
            ('811373', ' Jake G External (03.05.)', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('835144', 'Eric Sissan - Netherlands', 'en-US', 'Europe/Amsterdam', 'NL', 'No'),
            ('760642', 'External US number #1', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('661528', 'Netherlands - Nour Sadek', 'en-US', 'Etc/UTC', 'NL', 'No'),
            ('658868', 'Carolina mota', 'en-US', 'Etc/UTC', 'BR', 'No'),
            ('927643', 'Ashleigh Harpenau', 'en-US', 'America/Chicago', 'US', 'No'),
            ('661562', 'United Kingdom - Nikky Price', 'en-US', 'Etc/UTC', 'GB', 'No'),
            ('661552', 'United Kingdom - Eliza Arron', 'en-US', 'Etc/UTC', 'GB', 'No'),
            ('661560', 'United Kingdom - Max Cliffe', 'en-US', 'Etc/UTC', 'GB', 'No'),
            ('931833', 'Seleena Grewal', 'en-US', 'Europe/London', 'GB', 'No'),
            ('949136', 'number 3', 'en-US', 'Asia/Tokyo', 'JP', 'No'),
            ('949137', 'number 4', 'en-US', 'Asia/Tokyo', 'JP', 'No'),
            ('949138', 'number 5', 'en-US', 'Asia/Tokyo', 'JP', 'No'),
            ('949141', 'number 8', 'en-US', 'Asia/Tokyo', 'JP', 'No'),
            ('949145', 'number 12', 'en-US', 'Asia/Tokyo', 'JP', 'No'),
            ('949154', 'number 13', 'en-US', 'Asia/Tokyo', 'JP', 'No'),
            ('963821', 'Australia - Eric He', 'en-US', 'Australia/Sydney', 'AU', 'No'),
            ('957790', 'Nicholas Do - United States', 'en-US', 'America/Los_Angeles', 'US', 'No'),
            ('658636', '[Recycled UNUSED] Singapore number', 'en-US', 'Asia/Singapore', 'SG', 'No'),
            ('658637', 'SG number 2', 'en-US', 'Asia/Singapore', 'SG', 'No'),
            ('658109', 'Italy - Tommaso Caccia', 'en-US', 'Etc/UTC', 'IT', 'No'),
            ('658122', 'Berlin - Lina Weisenberg', 'en-US', 'Etc/UTC', 'DE', 'No'),
            ('660827', 'Netherlands - Stef Kennepohl', 'en-US', 'Etc/UTC', 'NL', 'No'),
            ('661548', 'Spain - Grazia Schiraldi', 'en-US', 'Etc/UTC', 'ES', 'No'),
            ('661550', 'Spain - Esteban Ollero', 'en-US', 'Etc/UTC', 'ES', 'No'),
            ('661573', 'United States - Sabine Rizvi', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('661575', 'United States - Paul Dupont', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('661591', 'United States - Nicole Roe', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('672531', 'Sweden - Felix Lantz', 'en-US', 'Etc/UTC', 'SE', 'No'),
            ('672539', 'France - Gregoire Leclercq', 'fr-FR', 'Etc/UTC', 'FR', 'No'),
            ('658865', 'Latam - Geovane Santos', 'en-US', 'Etc/UTC', 'BR', 'No'),
            ('989770', 'Ava Gueits', 'en-US', 'America/Los_Angeles', 'US', 'No'),
            ('707043', 'TEST_Spain L1', 'es-ES', 'Etc/UTC', 'ES', 'No'),
            ('976676', 'Dylan Silverstein', 'en-US', 'America/Chicago', 'US', 'No'),
            ('1019883', 'Lili Garcia', 'es-ES', 'America/Mexico_City', 'MX', 'No'),
            ('931219', 'Edward khouri ii', 'en-US', 'America/Chicago', 'US', 'No'),
            ('662908', 'Marine Revol', 'fr-FR', 'Europe/Paris', 'FR', 'No'),
            ('778477', 'Yesica Nava - United States', 'en-US', 'America/Chicago', 'US', 'No'),
            ('656631', 'SDR - Mexico City - Carolina Centeno', 'en-US', 'Etc/UTC', 'MX', 'No'),
            ('821463', 'External Mexico', 'es-ES', 'Etc/UTC', 'MX', 'No'),
            ('760764', 'External Number Mexico', 'en-US', 'Etc/UTC', 'MX', 'No'),
            ('722712', 'Madhav Gujral - United States', 'en-US', 'America/Chicago', 'US', 'No'),
            ('811381', 'Alison Andrade External (03.05.)', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('661594', 'United States - Alison Andrade', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('1019423', 'Esteba Ollero - Spain Mobile', 'es-ES', 'Europe/Madrid', 'ES', 'No'),
            ('902994', 'Google Play Store', 'en-US', 'Europe/Amsterdam', 'NL', 'No'),
            ('672537', 'Australia - Yen Bell', 'en-US', 'Etc/UTC', 'AU', 'No'),
            ('837434', 'Glauber Freire - Brazil External', 'en-US', 'Etc/UTC', 'BR', 'No'),
            ('760813', 'External Number Brazil', 'en-US', 'Etc/UTC', 'BR', 'No'),
            ('1060166', 'Eitan Dombey', 'en-US', 'America/New_York', 'US', 'No'),
            ('661592', 'United States - Colette Fernandes', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('661570', 'United States - Benjamin Bogard', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('660826', 'Netherlands - Stefano Canepa', 'en-US', 'Etc/UTC', 'NL', 'No'),
            ('710426', 'France mobile ', 'fr-FR', 'Etc/UTC', 'FR', 'No'),
            ('969227', 'Yoann SDR', 'en-US', 'America/Chicago', 'CA', 'No'),
            ('710421', 'Netherlands Mobile ', 'en-US', 'Etc/UTC', 'NL', 'No'),
            ('1101584', 'Jacobo', 'en-US', 'Europe/London', 'GB', 'No'),
            ('455304', 'HR/Recruitment APAC', 'en-US', 'Asia/Singapore', 'SG', 'No'),
            ('661576', 'United States - Marie Gabrielle de Woot', 'en-US', 'Etc/UTC', 'US', 'No'),
            ('1121779', 'Jakub Fraczek', 'en-US', 'Europe/Warsaw', 'PL', 'No'),
            ('1019888', 'Laura Romero', 'es-ES', 'America/Mexico_City', 'MX', 'No'),
            ('171410', '[Service]  Japan Disputes', 'en-US', 'Asia/Tokyo', 'JP', 'No'),
            ('658867', '[Service] Brazil Richemont L1 - V2', 'en-US', 'Etc/UTC', 'BR', 'Yes'),
            ('467595', '[Service] Canada', 'en-US', 'Europe/Paris', 'CA', 'Yes'),
            ('222498', '[Service] Canada L1', 'en-US', 'America/Chicago', 'CA', 'Yes'),
            ('477792', '[Service] France', 'en-US', 'Europe/Paris', 'FR', 'Yes'),
            ('465672', '[Service] Germany', 'de-DE', 'Europe/Amsterdam', 'DE', 'Yes'),
            ('222505', '[Service] Hong Kong L1', 'en-US', 'Asia/Hong_Kong', 'HK', 'Yes'),
            ('467609', '[Service] India Toll Free', 'en-US', 'Asia/Kolkata', 'IN', 'Yes'),
            ('475091', '[Service] Italy', 'it-IT', 'Europe/Amsterdam', 'IT', 'Yes'),
            ('222506', '[Service] Italy L1', 'it-IT', 'Europe/Rome', 'IT', 'Yes'),
            ('468140', '[Service] Malaysia', 'en-US', 'Asia/Kuala_Lumpur', 'MY', 'Yes'),
            ('467574', '[Service] Mexico', 'en-US', 'Europe/Paris', 'MX', 'Yes'),
            ('66720', '[Service] Portugal', 'pt-PT', 'Europe/Paris', 'PT', 'Yes'),
            ('467573', '[Service] Sweden', 'en-US', 'Europe/Stockholm', 'SE', 'Yes'),
            ('222510', '[Service] Sweden L1', 'en-US', 'Europe/Stockholm', 'SE', 'Yes'),
            ('467575', '[Service] Switzerland (French)', 'fr-FR', 'Europe/Zurich', 'CH', 'Yes'),
            ('507074', '[Service] UAE', 'en-US', 'Asia/Muscat', 'AE', 'Yes'),
            ('222514', '[Service] Switzerland L1', 'en-US', 'Europe/Zurich', 'CH', 'Yes'),
            ('759072', '[Service] UK Oracle L1', 'en-US', 'Europe/London', 'GB', 'Yes'),
            ('222513', '[Service] US L1', 'en-US', 'America/Chicago', 'US', 'Yes'),
            ('759066', '[Service] US Oracle L1', 'en-US', 'America/Chicago', 'US', 'Yes'),
            ('467570', '[Service] USA - SF', 'en-US', 'Europe/Paris', 'US', 'Yes'),
            ('943966', '[Service] Incidents', 'en-US', 'Europe/Amsterdam', 'NL', 'Yes'),
            ('321320', '[Service] Brazil', 'pt-PT', 'America/Sao_Paulo', 'BR', 'Yes'),
            ('1033701', 'EFP Sub-Merchant (NL)', 'en-US', 'Europe/Amsterdam', 'NL', 'No'),
            ('1148787', '❗ [Service] First Line Support', 'en-US', 'America/Los_Angeles', 'US', 'Yes'),
            ('467567', '[Service] London', 'en-US', 'Europe/Paris', 'GB', 'Yes'),
            ('222496', '[Service] Australia L1', 'en-US', 'Australia/Sydney', 'AU', 'Yes'),
            ('222497', '[Service] Belgium L1', 'fr-FR', 'Europe/Brussels', 'BE', 'Yes'),
            ('1201336', 'Lex Brown', 'en-US', 'America/New_York', 'US', 'No'),
            ('878400', 'Mathis Vuillermin', 'en-US', 'Etc/UTC', 'FR', 'No'),
            ('1219283', 'Sweden - Victor Lundell (SDR)', 'en-US', 'Europe/Stockholm', 'SE', 'No'),
            ('840912', 'French mobile number 1', 'en-US', 'Etc/UTC', 'FR', 'No')
    ]
    return spark.createDataFrame(data=data, schema=schema)

In [32]:
call_mapping_df = create_numbers_mapping(spark)
call_mapping_df.show(5)

915244,Seleena Grewal,en-US,Etc/UTC,GB,No
922734,Melanie Vecchio,en-US,Europe/Paris,FR,No
721818,Gaspard Le Poittevin - France,fr-FR,Etc/UTC,FR,No
722709,Jake Guenoun - United States,en-US,America/Chicago,US,No
726382,Felix Lantz - Switzerland Mobile,en-US,Europe/Stockholm,SE,No


In [40]:
def bdp_config_calls(df, call_mapping_df):
    df = df.join(call_mapping_df, "number", how='left')
    
    df = df.withColumn('month', F.date_format(F.date_trunc('MM', F.col('started_at')), 'yyyy-MM-dd'))
    
    df = df.withColumn(
        'support_team',
        F.when(F.col('name').contains('Japan Disputes'), 'Japan - Disputes')
         .when(F.col('name').contains('Japan'), 'Japan - Operational Support')
         .when(F.col('name').contains('Brazil') | F.col('name').contains('BR Duty'), 'LATAM - Operational Support')
         .otherwise("Operational Support") 
    )
    
    filtered_df = df.filter(
        (F.col('name').isNotNull()) & 
        (F.col('name').startswith('[Service]')) &
        (F.col('name') != '[Service]  POS 2ndLine') &
        (F.col('direction').isin('inbound', 'outbound')) &
        (F.year(F.col('started_at')) >= 2023) &
        (F.col('answered_at').isNotNull()) 
    )
    
    result_df = filtered_df.groupBy('month', 'support_team') \
        .agg(F.count('*').alias('calls')) \
        .orderBy('month', 'support_team')
    
    return result_df

In [41]:
clean_bdp_calls_df = bdp_config_calls(calls_df,call_mapping_df)
clean_bdp_calls_df.show(10)

2025-10-01,Japan - Disputes,98
2025-10-01,Japan - Operational Support,99
2025-10-01,LATAM - Operational Support,70
2025-10-01,Operational Support,1958
2025-11-01,Japan - Disputes,111
2025-11-01,Japan - Operational Support,77
2025-11-01,LATAM - Operational Support,74
2025-11-01,Operational Support,1893
2025-12-01,Japan - Disputes,137
2025-12-01,Japan - Operational Support,133


In [42]:
clean_bdp_calls_df.select("support_team").distinct().show()

Japan - Operational Support
LATAM - Operational Support
Operational Support
Japan - Disputes


In [ ]:
clean_bdp_calls_df.write.mode("overwrite").parquet(f"{output_directory}/clean_calls_df.parquet")